# Phase 2 — Random Forest Training

Trains a Random Forest classifier to predict the offloading **target**
(LOCAL / EDGE / CLOUD) from the device-context features logged by
MetricsRecorder.

**Inputs**: `../data/training.csv` (collected via Phase 1 runbook)
**Outputs**:
- `../outputs/feature-importance.png`
- `../outputs/confusion-matrix.png`
- `../outputs/comparison-summary.md`
- `../outputs/rf-model.json`  (deployable to Android)

Run cells top-to-bottom. Re-run cell 5 (Feature engineering) if you
change feature selection. Re-run cell 7 (RF training) to retrain.

## 1 · Imports

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder

DATA_PATH = Path('../data/training.csv')
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
print('Setup ok')

## 2 · Load CSV

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Rows: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head()

## 3 · Exploratory data analysis

Sanity-check the data before training:
- Target distribution (LOCAL / EDGE / CLOUD)
- Rule distribution (all 7 represented?)
- Numeric feature ranges

In [ ]:
print('--- Target distribution ---')
print(df['target'].value_counts())
print()
print('--- Rule distribution (excluding baseline modes) ---')
adaptive = df[~df['rule'].str.startswith('FORCED_')]
print(adaptive['rule'].value_counts())
print()
print('--- Numeric summary ---')
adaptive.select_dtypes(include=[np.number]).describe()

## 4 · Filter to adaptive rows only

Baseline modes (`FORCED_LOCAL`, `FORCED_CLOUD`) are kept in the CSV for
the latency-comparison plots later but **excluded from RF training** —
the model should learn the *rule-based policy*, not the forced modes.

In [ ]:
train_df = df[~df['rule'].str.startswith('FORCED_')].copy()
print(f'Adaptive rows kept for training: {len(train_df)}')
print(f'Baseline rows reserved for comparison: {len(df) - len(train_df)}')

## 5 · Feature engineering

Selected features (no leakage — we exclude `target`, `rule`, `reasoning`,
`actual_ms`, and identifier columns).

Encoding:
- `network_type`: ordinal (NONE=0, LTE=1, WIFI=2, 5G=3)
- `task_complexity`: derived from `task_name` then ordinal (LIGHT=0, MEDIUM=1, HEAVY=2)
- booleans: 0/1
- continuous numerics: left as-is

In [ ]:
NETWORK_RANK = {'NONE': 0, 'LTE': 1, 'WIFI': 2, '5G': 3}
TASK_COMPLEXITY = {
    'echo': 0, 'sha256': 0,                   # LIGHT
    'image-grayscale': 1,                     # MEDIUM
    'matrix-multiply': 2, 'video-frame-edges': 2,  # HEAVY
}

def engineer(d):
    out = pd.DataFrame()
    out['battery_percent'] = d['battery_percent']
    out['is_charging'] = d['is_charging'].astype(int)
    out['network_type_rank'] = d['network_type'].map(NETWORK_RANK).fillna(0)
    out['network_score'] = d['network_score']
    out['rtt_ms'] = d['rtt_ms']
    out['bandwidth_mbps'] = d['bandwidth_mbps']
    out['cpu_percent'] = d['cpu_percent']
    out['is_stable'] = d['is_stable'].astype(int)
    out['task_complexity'] = d['task_name'].map(TASK_COMPLEXITY).fillna(-1)
    out['est_local_ms'] = d['est_local_ms']
    out['est_remote_ms'] = d['est_remote_ms']
    out['speedup'] = d['speedup']
    return out

X = engineer(train_df)
y = train_df['target']

FEATURE_NAMES = list(X.columns)
print('Features:', FEATURE_NAMES)
X.head()

## 6 · Train / test split

Stratified by `target` so each class is represented in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE,
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')
print('Train class balance:')
print(y_train.value_counts(normalize=True).round(2))

## 7 · Random Forest training

Hyperparameters:
- `n_estimators=50` — enough for stability, still lightweight for on-device export
- `max_depth=8` — caps tree size (each tree exportable to JSON cleanly)
- `class_weight='balanced'` — counteracts uneven rule distribution
- `random_state` fixed for reproducibility

In [ ]:
rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=8,
    min_samples_leaf=3,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
print('Training done')
print(f'Train accuracy: {rf.score(X_train, y_train):.3f}')
print(f'Test  accuracy: {rf.score(X_test, y_test):.3f}')

cv_scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy', n_jobs=-1)
print(f'5-fold CV accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

## 8 · Classification report

In [ ]:
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

## 9 · Confusion matrix

Where does the RF disagree with the rule-based ground truth?

In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel('Predicted (RF)')
ax.set_ylabel('Actual (rule-based ground truth)')
ax.set_title('Random Forest vs rule-based — confusion matrix')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'confusion-matrix.png', dpi=160)
plt.show()

## 10 · Feature importance

Which device-context signals matter most for predicting the target?

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURE_NAMES)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 5))
importances.plot.barh(ax=ax, color='#1F4E79')
ax.set_xlabel('Importance (Gini)')
ax.set_title('RF feature importance')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'feature-importance.png', dpi=160)
plt.show()

print(importances.sort_values(ascending=False).to_string())

## 11 · Latency comparison — RF vs rule-based vs baselines

Side-by-side using `actual_ms` from CSV (per task complexity).

In [ ]:
# Tag each row with which policy produced it
df['policy'] = df['rule'].apply(
    lambda r: 'Local-only' if r == 'FORCED_LOCAL'
    else 'Cloud-only' if r == 'FORCED_CLOUD'
    else 'Rule-based'
)

summary = df.groupby(['policy', 'task_name'])['actual_ms'].agg(
    ['mean', 'std', 'count']
).round(1)
print(summary)

## 12 · Export model for on-device deployment

Serialise each tree to a portable JSON structure (`rf-model.json`) so
the Android Kotlin runtime can load and predict without any ML framework.

Format:
```
{
  "feature_names": [...],
  "classes": ["CLOUD", "EDGE", "LOCAL"],
  "trees": [
    {
      "feature":   [int, ...],   # -1 for leaf nodes
      "threshold": [float, ...],
      "left":      [int, ...],
      "right":     [int, ...],
      "value":     [[float, float, float], ...]   # class probabilities (leaf only)
    },
    ...
  ]
}
```

In [ ]:
def tree_to_dict(tree):
    t = tree.tree_
    # Normalise leaf values to class probabilities
    values = t.value.reshape(-1, len(rf.classes_))
    values = values / values.sum(axis=1, keepdims=True)
    return {
        'feature':   t.feature.astype(int).tolist(),
        'threshold': t.threshold.astype(float).tolist(),
        'left':      t.children_left.astype(int).tolist(),
        'right':     t.children_right.astype(int).tolist(),
        'value':     values.astype(float).tolist(),
    }

model_json = {
    'feature_names': FEATURE_NAMES,
    'classes':       list(rf.classes_),
    'network_rank':  NETWORK_RANK,
    'task_complexity': TASK_COMPLEXITY,
    'trees':         [tree_to_dict(t) for t in rf.estimators_],
}

model_path = OUTPUT_DIR / 'rf-model.json'
with model_path.open('w') as f:
    json.dump(model_json, f)

size_kb = model_path.stat().st_size / 1024
print(f'Exported {model_path.name} — {size_kb:.1f} KB')
print(f'Trees: {len(model_json["trees"])}, classes: {model_json["classes"]}')

## 13 · Sanity-check exported model

Reload the JSON and predict on a few rows. If predictions match
`rf.predict(X_test)` exactly, the exported model is byte-identical for
deterministic inputs and safe to ship to Android.

In [ ]:
def predict_with_json(model, x_row):
    votes = np.zeros(len(model['classes']))
    for tree in model['trees']:
        node = 0
        while tree['feature'][node] >= 0:  # not a leaf
            f = tree['feature'][node]
            t = tree['threshold'][node]
            node = tree['left'][node] if x_row[f] <= t else tree['right'][node]
        votes += np.array(tree['value'][node])
    return model['classes'][int(np.argmax(votes))]

json_preds = [predict_with_json(model_json, X_test.iloc[i].values)
              for i in range(min(20, len(X_test)))]
rf_preds   = list(rf.predict(X_test.iloc[:len(json_preds)]))
match = sum(j == r for j, r in zip(json_preds, rf_preds))
print(f'JSON-model predictions match sklearn: {match}/{len(json_preds)}')

## 14 · Comparison summary

Write a short markdown summary to `comparison-summary.md` for the thesis.
Includes accuracy, top features, and latency-per-task table.

In [ ]:
lines = [
    '# RF vs Rule-based — Summary',
    '',
    f'- Adaptive rows used: **{len(train_df)}**',
    f'- Test accuracy: **{rf.score(X_test, y_test):.3f}**',
    f'- 5-fold CV accuracy: **{cv_scores.mean():.3f} ± {cv_scores.std():.3f}**',
    '',
    '## Top-5 features',
    *[f'- {n}: {v:.3f}'
      for n, v in importances.sort_values(ascending=False).head(5).items()],
    '',
    '## Latency by policy and task (mean ms)',
    '',
    '```',
    summary.to_string(),
    '```',
]

out = OUTPUT_DIR / 'comparison-summary.md'
out.write_text('\n'.join(lines), encoding='utf-8')
print(f'Wrote {out}')